# Model Approval Check

This notebook checks if a model version has been approved for deployment.

**Purpose:**
- Verify that the model version has the approval tag
- Check that the approval tag value is 'approved'
- Block deployment if not approved

**Note:** This notebook should only be run in a Databricks Job, as part of MLflow 3.0 Deployment Jobs.

## Setup

In [0]:
from mlflow import MlflowClient

In [0]:
# Define widgets for job parameters
dbutils.widgets.text("model_name", "")
dbutils.widgets.text("model_version", "")
dbutils.widgets.text("approval_tag_name", "deployment_approval")

In [0]:
# Get parameters
model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
approval_tag_name = dbutils.widgets.get("approval_tag_name")

print(f"Checking Approval for Model: {model_name}")
print(f"Version: {model_version}")
print(f"Approval Tag: {approval_tag_name}")

## Check Approval Status

In [0]:
# Initialize MLflow client for Unity Catalog
client = MlflowClient(registry_uri="databricks-uc")

# Get model version details
model_version_details = client.get_model_version(model_name, model_version)

print(f"\nModel Version Details:")
print(f"  Name: {model_version_details.name}")
print(f"  Version: {model_version_details.version}")
print(f"  Status: {model_version_details.status}")

In [0]:
# Fetch the model version's Unity Catalog tags
tags = model_version_details.tags

print(f"\nAll tags on model version:")
for tag_key, tag_value in tags.items():
    print(f"  {tag_key}: {tag_value}")

## Verify Approval

In [0]:
# Check if approval tag exists
if approval_tag_name not in tags:
    error_msg = f"Model version not approved for deployment. Missing tag: '{approval_tag_name}'"
    print(f"\n❌ {error_msg}")
    print(f"\nTo approve this model, add the tag '{approval_tag_name}' with value 'approved' in the UI or via API.")
    raise Exception(error_msg)

# Check if approval tag value is 'approved'
approval_value = tags.get(approval_tag_name).lower()
if approval_value != "approved":
    error_msg = f"Model version not approved for deployment. Tag '{approval_tag_name}' = '{approval_value}' (expected 'approved')"
    print(f"\n❌ {error_msg}")
    raise Exception(error_msg)

# If we reach here, the model is approved
print(f"\n✓ Model version approved for deployment")
print(f"  Tag: {approval_tag_name} = {approval_value}")

## Summary

In [0]:
print("\n" + "="*60)
print("APPROVAL CHECK PASSED")
print("="*60)
print(f"Model: {model_name}")
print(f"Version: {model_version}")
print(f"Approval Tag: {approval_tag_name} = approved")
print("="*60)
print("\n✓ Model is approved. Proceeding to deployment step.")